In [1]:
# ================================================================
# NeurIPS OPP 2025 — Mordred + Chem-kNN + Stacking (Fixed & Tuned)
# ================================================================

import os, sys, gc, subprocess, numpy as np, pandas as pd
from pathlib import Path

# ----------------------------
# Utilities: controlled pip
# ----------------------------
def _maybe_pip(path):
    if os.path.exists(path):
        try:
            print(f"[Install] {os.path.basename(path)}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", path])
        except Exception as e:
            print(f"[Install skipped] {os.path.basename(path)} -> {e}")

# ----------------------------
# Wheels (adjust if paths differ)
# ----------------------------
RDKit_WHL_DIR = "/kaggle/input/rdkit-2025-3-3-cp311"
MORDRED_DIR   = "/kaggle/input/mordred-1-2-0-py3-none-any"

NX_WHL       = f"{MORDRED_DIR}/networkx-2.8.8-py3-none-any.whl"
MORDRED_WHL  = f"{MORDRED_DIR}/mordred-1.2.0-py3-none-any.whl"
RDKit_WHL    = f"{RDKit_WHL_DIR}/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl"

_maybe_pip(NX_WHL)
_maybe_pip(RDKit_WHL)
_maybe_pip(MORDRED_WHL)

# ----------------------------
# Imports
# ----------------------------
from rdkit import Chem, DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from mordred import Calculator, descriptors

from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

CATBOOST_OK = False
try:
    from catboost import CatBoostRegressor
    CATBOOST_OK = True
except Exception as e:
    print("[CatBoost unavailable]", e)

HAS_LGBM = False
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    pass

try:
    from scipy.optimize import nnls
    HAS_NNLS = True
except Exception:
    HAS_NNLS = False

# ----------------------------
# Data discovery
# ----------------------------
def _find_data_dir():
    cands = [d for d in Path("/kaggle/input").glob("*")]
    key = lambda s: any(k in s.lower() for k in ["polymer", "neurips", "open", "opp"])
    for d in cands:
        if d.is_dir() and key(d.name.lower()):
            if (d/"train.csv").exists() and (d/"test.csv").exists():
                return str(d)
    return "/kaggle/input/neurips-open-polymer-prediction-2025"

DATA_DIR = _find_data_dir()
MODRED_TRAIN_DESC_DIR = "/kaggle/input/modred-dataset"
print("[DATA_DIR]", DATA_DIR)
print("[MODRED_TRAIN_DESC_DIR]", MODRED_TRAIN_DESC_DIR)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

def _find_smiles(df):
    for c in df.columns:
        if str(c).lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("SMILES column not found in dataframe")

SMI_COL = _find_smiles(train)
TARGETS = ["Tg","FFV","Tc","Density","Rg"]

RANDOM_STATE = 42
N_SPLITS = 5

# ----------------------------
# Descriptor cache (TEST)
# ----------------------------
CACHE_TEST_DESC = "/kaggle/working/mordred_test.csv"
if os.path.exists(CACHE_TEST_DESC):
    desc_test = pd.read_csv(CACHE_TEST_DESC)
    desc_test = desc_test.loc[:, ~desc_test.columns.str.contains("^Unnamed")]
    print("[Cache] Loaded test descriptors:", desc_test.shape)
else:
    print("[Mordred] Computing test descriptors (2D; ignore_3D=True)...")
    mols_test = [Chem.MolFromSmiles(s) for s in test[SMI_COL].astype(str).tolist()]
    calc = Calculator(descriptors, ignore_3D=True)
    # mordred DataFrame contains mixed dtypes; we'll coerce later
    desc_test = calc.pandas(mols_test)
    # Coerce numeric where possible
    desc_test = desc_test.apply(pd.to_numeric, errors="coerce")
    desc_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    desc_test.to_csv(CACHE_TEST_DESC, index=False)
    print("[Cache] Saved test descriptors:", desc_test.shape)

# ----------------------------
# Cleaning helpers
# ----------------------------
def clean_desc_df(df, target_name=None):
    df = df.copy()
    for c in df.columns:
        if c != target_name:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    # Drop constant / all-NaN cols except target
    nunique = df.nunique(dropna=True)
    drop_cols = [c for c in df.columns if c != target_name and (nunique.get(c, 2) <= 1)]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

# De-dupe OOF (fixes "cannot reindex with duplicate labels")
def _dedupe_oof(idx_vec, oof_vec, how="mean"):
    s = pd.Series(np.asarray(oof_vec, dtype=float),
                  index=pd.Index(np.asarray(idx_vec), name="train_idx"))
    if s.index.has_duplicates:
        if how == "median":
            s = s.groupby(level=0).median()
        else:
            s = s.groupby(level=0).mean()
    else:
        s = s.astype(float)
    return s.index.to_numpy(), s.values

def _align_oof_matrix(bases, idx_sorted):
    """bases: list of (name, oof_vec, idx_vec, tst_vec) with unique idx_vec"""
    X_oof_cols, X_tst_cols, base_names = [], [], []
    for name, oof_vec, idx_vec, tst_vec in bases:
        col = pd.Series(oof_vec, index=pd.Index(idx_vec, name="train_idx"))
        X_oof_cols.append(col.reindex(idx_sorted).values)
        X_tst_cols.append(np.asarray(tst_vec, dtype=float))
        base_names.append(name)
    return np.column_stack(X_oof_cols), np.column_stack(X_tst_cols), base_names

def robust_clip(pred, y_ref):
    y = pd.Series(y_ref).dropna().values
    if len(y) < 10:
        return pred
    q25, q75, q99 = np.nanpercentile(y, [25, 75, 99])
    iqr = q75 - q25
    lo = q25 - 1.5 * iqr
    hi = q99  # slight upper clamp
    return np.clip(pred, lo, hi)

# ----------------------------
# Models
# ----------------------------
def make_catboost_cpu():
    return CatBoostRegressor(
        loss_function="MAE",
        depth=8,
        learning_rate=0.05,
        iterations=3000,             # OD will pick best
        l2_leaf_reg=4.0,
        random_seed=RANDOM_STATE,
        verbose=False,
        od_type="Iter",
        od_wait=300,
        thread_count=-1,             # use all CPUs
    )

def make_lgbm():
    return LGBMRegressor(
        objective="mae",
        n_estimators=4000,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

# ----------------------------
# Canonical SMILES (for grouping & fill)
# ----------------------------
def canonize(smiles):
    m = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(m, canonical=True) if m is not None else None

can_train = train[SMI_COL].astype(str).map(canonize)
can_test  = test[SMI_COL].astype(str).map(canonize)

# ----------------------------
# Mordred training paths
# ----------------------------
DESC_PATHS = {
    "Tg":      f"{MODRED_TRAIN_DESC_DIR}/desc_tg.csv",
    "FFV":     f"{MODRED_TRAIN_DESC_DIR}/desc_ffv.csv",
    "Tc":      f"{MODRED_TRAIN_DESC_DIR}/desc_tc.csv",
    "Density": f"{MODRED_TRAIN_DESC_DIR}/desc_de.csv",
    "Rg":      f"{MODRED_TRAIN_DESC_DIR}/desc_rg.csv",
}
DESC_PATHS = {t:p for t,p in DESC_PATHS.items() if os.path.exists(p)}

# ----------------------------
# Mordred fit/predict with GroupKFold by canonical SMILES
# ----------------------------
def fit_predict_mordred(target, desc_train_path, desc_test_df, return_oof=True):
    print(f"\n=== [Mordred/{target}] using {os.path.basename(desc_train_path)} ===")
    raw = pd.read_csv(desc_train_path, low_memory=False)

    # Identify mapping cols on the RAW frame (before any drops)
    def _find_col(df, names):
        names = [n.lower() for n in names]
        for c in df.columns:
            if str(c).lower().strip() in names:
                return c
        return None

    DESC_ID_COL  = _find_col(raw, ["id"])
    DESC_SMI_COL = _find_col(raw, ["smiles","smile","smiles_str","polymer_smiles"])

    # Map raw rows to train indices (to get grouping by canonical SMILES)
    train_idx_map = None
    try:
        if DESC_ID_COL is not None and "id" in train.columns:
            j = raw[[DESC_ID_COL]].reset_index().rename(columns={"index":"desc_row"})
            j = j.merge(train[["id"]].reset_index().rename(columns={"index":"train_idx"}),
                        left_on=DESC_ID_COL, right_on="id", how="left")
            train_idx_map = j["train_idx"].to_numpy()
        elif DESC_SMI_COL is not None:
            j = raw[[DESC_SMI_COL]].reset_index().rename(columns={"index":"desc_row"})
            tr_map = train[[SMI_COL]].reset_index().rename(columns={"index":"train_idx"})
            j = j.merge(tr_map, left_on=DESC_SMI_COL, right_on=SMI_COL, how="left")
            train_idx_map = j["train_idx"].to_numpy()
    except Exception as e:
        print(f"[Mordred/{target}] train-index mapping skipped: {e}")
        train_idx_map = None

    train_d = clean_desc_df(raw, target_name=target)
    test_d  = clean_desc_df(desc_test_df)

    drop_nonfeat = {c for c in [target, DESC_ID_COL, DESC_SMI_COL] if c is not None}
    feat_cols = sorted(list((set(train_d.columns) - drop_nonfeat) & set(test_d.columns)))
    X = train_d[feat_cols]
    y = train_d[target].astype(float)
    X_test = test_d[feat_cols]

    use_lgbm = (not CATBOOST_OK) and HAS_LGBM
    if use_lgbm:
        imp = SimpleImputer(strategy="median")
        X = pd.DataFrame(imp.fit_transform(X), columns=feat_cols)
        X_test = pd.DataFrame(imp.transform(X_test), columns=feat_cols)

    # Build groups if we have a mapping into the original train
    groups = None
    if train_idx_map is not None:
        try:
            # Some rows may not map; fill NA with raw SMILES canonicalization if present
            groups = pd.Series(train_idx_map).map(can_train)
            if groups.isna().any() and DESC_SMI_COL is not None:
                # fallback to canonicalized raw smiles when needed
                can_raw = raw[DESC_SMI_COL].astype(str).map(canonize)
                groups = groups.fillna(can_raw)
            groups = groups.fillna("NA").values
        except Exception as e:
            print(f"[Mordred/{target}] Group fallback to KFold due to: {e}")
            groups = None

    splitter = GroupKFold(n_splits=N_SPLITS) if groups is not None else KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    oof = np.zeros(len(X), dtype=float) if return_oof else None
    pred_test = np.zeros(len(X_test), dtype=float)

    for fold, split in enumerate(splitter.split(X, y, groups=groups) if groups is not None else splitter.split(X), 1):
        tr_idx, va_idx = split
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]

        if use_lgbm:
            model = make_lgbm()
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l1", verbose=False)
        else:
            model = make_catboost_cpu()
            model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True, verbose=False)

        if return_oof:
            oof[va_idx] = model.predict(Xva)
        pred_test += model.predict(X_test) / N_SPLITS

        del model, Xtr, Xva, ytr, yva
        gc.collect()

    if return_oof:
        mae = float(mean_absolute_error(y, oof))
        print(f"[Mordred/{target}] OOF MAE: {mae:.6f} (features: {len(feat_cols)})")

    return (oof if return_oof else None), pred_test, y.values, train_idx_map

# ----------------------------
# Run Mordred bases
# ----------------------------
OOF_MORDRED, TEST_MORDRED, Y_MORDRED, IDX_MORDRED = {}, {}, {}, {}
for t in TARGETS:
    if t in DESC_PATHS:
        oof, pred_t, y_t, idx_map = fit_predict_mordred(t, DESC_PATHS[t], desc_test, return_oof=True)
        OOF_MORDRED[t] = oof
        TEST_MORDRED[t] = pred_t
        Y_MORDRED[t] = y_t
        IDX_MORDRED[t] = idx_map
    else:
        print(f"[Mordred/{t}] descriptor CSV not found; skipping.")

# ----------------------------
# Chem-kNN (Morgan r=2,3) with GroupKFold
# ----------------------------
print("\n[Chem-kNN] Building Morgan fingerprints (r=2 & r=3) ...")
N_BITS = 2048
gen2 = GetMorganGenerator(radius=2, fpSize=N_BITS)
gen3 = GetMorganGenerator(radius=3, fpSize=N_BITS)

def fps_from_smiles(series, gen):
    mols = [Chem.MolFromSmiles(s) for s in series.astype(str).tolist()]
    return [gen.GetFingerprint(m) if m is not None else None for m in mols]

fps2_train_all = fps_from_smiles(train[SMI_COL], gen2)
fps3_train_all = fps_from_smiles(train[SMI_COL], gen3)
fps2_test_all  = fps_from_smiles(test[SMI_COL],  gen2)
fps3_test_all  = fps_from_smiles(test[SMI_COL],  gen3)

def knn_predict_one(fp, fps_train, y_train, topk=32, poww=2.0):
    if fp is None or len(fps_train) == 0:
        return np.nan
    sims = DataStructs.BulkTanimotoSimilarity(fp, fps_train)
    if len(sims) == 0:
        return np.nan
    k = min(topk, len(sims))
    idx = np.argpartition(sims, -k)[-k:]
    w = np.asarray(sims, dtype=float)[idx]
    y = y_train[idx]
    w = np.maximum(w, 1e-12) ** poww
    return float(np.sum(w * y) / np.sum(w))

def knn_oof_and_test_with_idx(target, topk=32, poww=2.0, use_both_radii=True):
    mask = train[target].notnull().values
    idx_mask = np.flatnonzero(mask)     # train indices used
    y = train.loc[mask, target].values.astype(float)
    groups = can_train[mask].fillna("NA").values

    fps2_train = [fps2_train_all[i] for i in idx_mask]
    fps3_train = [fps3_train_all[i] for i in idx_mask]

    gkf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros_like(y, dtype=float)

    for tr, va in gkf.split(np.zeros(len(y)), groups=groups):
        y_tr = y[tr]
        fps2_tr = [fps2_train[i] for i in tr]
        fps3_tr = [fps3_train[i] for i in tr]
        for j, idx in enumerate(va):
            p2 = knn_predict_one(fps2_train[idx], fps2_tr, y_tr, topk=topk, poww=poww)
            if use_both_radii:
                p3 = knn_predict_one(fps3_train[idx], fps3_tr, y_tr, topk=topk, poww=poww)
                oof[va[j]] = np.nanmean([p2, p3])
            else:
                oof[va[j]] = p2

    preds_test = []
    for fp2, fp3 in zip(fps2_test_all, fps3_test_all):
        p2 = knn_predict_one(fp2, fps2_train, y, topk=topk, poww=poww)
        if use_both_radii:
            p3 = knn_predict_one(fp3, fps3_train, y, topk=topk, poww=poww)
            preds_test.append(np.nanmean([p2, p3]))
        else:
            preds_test.append(p2)

    return oof, np.array(preds_test, dtype=float), y, idx_mask

# Light grid everywhere, slightly richer for Density only
KNN_TOPK_GRID = {
    "Tg":      [16, 24, 32],
    "FFV":     [24, 32, 48],
    "Tc":      [24, 32, 48],
    "Density": [20, 24, 32],  # add 20 for a sharper neighborhood
    "Rg":      [16, 24, 32],
}
KNN_POWW_GRID = {
    "Density": [2.0, 3.0],    # try sharper weights only for Density
}

OOF_KNN, TEST_KNN, Y_KNN, IDX_KNN = {}, {}, {}, {}
for t in TARGETS:
    if t not in train.columns:
        continue
    grid_topk = KNN_TOPK_GRID.get(t, [32])
    grid_poww = KNN_POWW_GRID.get(t, [2.0])

    best = None
    for poww in grid_poww:
        for k in grid_topk:
            oof, pred, y_k, idx_mask = knn_oof_and_test_with_idx(t, topk=k, poww=poww, use_both_radii=True)
            mae = mean_absolute_error(y_k, oof)
            print(f"[Chem-kNN/{t}] TOPK={k} POWW={poww} -> OOF MAE={mae:.6f}")
            if (best is None) or (mae < best[0]):
                best = (mae, k, poww, oof, pred, y_k, idx_mask)

    mae, k, poww, oof, pred, y_k, idx_mask = best
    print(f"[Chem-kNN/{t}] Selected TOPK={k} POWW={poww} (OOF MAE={mae:.6f})")
    OOF_KNN[t] = oof
    TEST_KNN[t] = pred
    Y_KNN[t] = y_k
    IDX_KNN[t] = idx_mask

# ----------------------------
# Stacking with de-dupe fix
# ----------------------------
PRED_STACK = pd.DataFrame({"id": test["id"].values})

for t in TARGETS:
    base_cols = []
    idx_sets = []

    # Mordred base (with index map => de-dupe)
    if t in OOF_MORDRED and t in TEST_MORDRED and IDX_MORDRED.get(t, None) is not None:
        m_idx_raw = IDX_MORDRED[t]
        if m_idx_raw is not None:
            valid_m = ~pd.isna(m_idx_raw)
            m_idx_raw = m_idx_raw[valid_m].astype(int)
            m_oof_raw = OOF_MORDRED[t][valid_m]
            m_idx, m_oof = _dedupe_oof(m_idx_raw, m_oof_raw, how="mean")
            base_cols.append(("mordred", m_oof, m_idx, TEST_MORDRED[t]))
            idx_sets.append(set(m_idx))

    # Chem-kNN base (ensure unique indices)
    if t in OOF_KNN and t in TEST_KNN and t in IDX_KNN:
        k_idx_raw = IDX_KNN[t].astype(int)
        k_oof_raw = OOF_KNN[t]
        k_idx, k_oof = _dedupe_oof(k_idx_raw, k_oof_raw, how="mean")
        base_cols.append(("knn", k_oof, k_idx, TEST_KNN[t]))
        idx_sets.append(set(k_idx))

    # If no bases (shouldn't happen), fallback to Mordred-only test preds
    if not base_cols:
        print(f"[Stack/{t}] No bases; using Mordred-only if present.")
        pred = TEST_MORDRED.get(t, np.zeros(len(test), dtype=float))
        PRED_STACK[t] = robust_clip(pred, train[t].dropna().values if t in train.columns else pred)
        continue

    # Align on intersection of TRAIN indices across bases (now unique)
    idx_intersection = set.intersection(*idx_sets) if idx_sets else set()
    if len(idx_intersection) < 5:
        print(f"[Stack/{t}] Alignment small (|intersect|={len(idx_intersection)}); using Mordred-only.")
        pred = TEST_MORDRED.get(t, np.zeros(len(test), dtype=float))
        PRED_STACK[t] = robust_clip(pred, train[t].dropna().values if t in train.columns else pred)
        continue

    idx_sorted = np.array(sorted(idx_intersection), dtype=int)
    X_oof, X_tst, base_names = _align_oof_matrix(base_cols, idx_sorted)
    y_aligned = train.loc[idx_sorted, t].values.astype(float)

    # Fit stacker
    if HAS_NNLS:
        w, _ = nnls(X_oof, y_aligned)
        s = w.sum()
        if s > 0:
            w = w / s
        print(f"[Stack/{t}] NNLS weights {dict(zip(base_names, np.round(w,4)))}")
        pred = X_tst @ w
    else:
        stk = Ridge(alpha=1e-2, random_state=RANDOM_STATE)
        stk.fit(X_oof, y_aligned)
        print(f"[Stack/{t}] Ridge stacker coefs {dict(zip(base_names, np.round(stk.coef_,4)))}")
        pred = stk.predict(X_tst)

    pred = robust_clip(pred, y_aligned)
    PRED_STACK[t] = pred

# ----------------------------
# Fill exact SMILES matches by mean of train duplicates
# ----------------------------
for t in TARGETS:
    if t not in train.columns:
        continue
    m = train[t].notnull()
    means = (pd.DataFrame({ "can": can_train[m].values, t: train.loc[m, t].values})
             .groupby("can", as_index=True)[t].mean())
    match = can_test.isin(means.index)
    if match.any():
        fill_vals = can_test[match].map(means).values
        PRED_STACK.loc[match, t] = fill_vals

# ----------------------------
# Save submission
# ----------------------------
sub = pd.DataFrame({
    "id":      PRED_STACK["id"].values,
    "Tg":      PRED_STACK.get("Tg", pd.Series(np.zeros(len(test)))).values,
    "FFV":     PRED_STACK.get("FFV", pd.Series(np.zeros(len(test)))).values,
    "Tc":      PRED_STACK.get("Tc", pd.Series(np.zeros(len(test)))).values,
    "Density": PRED_STACK.get("Density", pd.Series(np.zeros(len(test)))).values,
    "Rg":      PRED_STACK.get("Rg", pd.Series(np.zeros(len(test)))).values,
})
out_path = "/kaggle/working/submission.csv"
sub.to_csv(out_path, index=False)
print("[Saved]", out_path)

# ----------------------------
# Optional diagnostics
# ----------------------------
for t in TARGETS:
    if t in IDX_MORDRED and IDX_MORDRED[t] is not None:
        vc = pd.Series(IDX_MORDRED[t]).value_counts()
        dups = int((vc > 1).sum())
        if dups:
            print(f"[Diag/{t}] duplicate train indices in Mordred mapping: {dups}")


[Install] networkx-2.8.8-py3-none-any.whl


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_

[Install] rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
[Install] mordred-1.2.0-py3-none-any.whl
[DATA_DIR] /kaggle/input/neurips-open-polymer-prediction-2025
[MODRED_TRAIN_DESC_DIR] /kaggle/input/modred-dataset
[Mordred] Computing test descriptors (2D; ignore_3D=True)...


100%|██████████| 3/3 [00:00<00:00,  5.03it/s]


[Cache] Saved test descriptors: (3, 1613)

=== [Mordred/Tg] using desc_tg.csv ===
[Mordred/Tg] OOF MAE: 28.510940 (features: 506)

=== [Mordred/FFV] using desc_ffv.csv ===
[Mordred/FFV] OOF MAE: 0.006135 (features: 506)

=== [Mordred/Tc] using desc_tc.csv ===
[Mordred/Tc] OOF MAE: 0.031473 (features: 506)

=== [Mordred/Density] using desc_de.csv ===
[Mordred/Density] OOF MAE: 0.065797 (features: 506)

=== [Mordred/Rg] using desc_rg.csv ===
[Mordred/Rg] OOF MAE: 1.587054 (features: 506)

[Chem-kNN] Building Morgan fingerprints (r=2 & r=3) ...
[Chem-kNN/Tg] TOPK=16 POWW=2.0 -> OOF MAE=52.864206
[Chem-kNN/Tg] TOPK=24 POWW=2.0 -> OOF MAE=52.905584
[Chem-kNN/Tg] TOPK=32 POWW=2.0 -> OOF MAE=53.373444
[Chem-kNN/Tg] Selected TOPK=16 POWW=2.0 (OOF MAE=52.864206)
[Chem-kNN/FFV] TOPK=24 POWW=2.0 -> OOF MAE=0.010194
[Chem-kNN/FFV] TOPK=32 POWW=2.0 -> OOF MAE=0.010579
[Chem-kNN/FFV] TOPK=48 POWW=2.0 -> OOF MAE=0.011139
[Chem-kNN/FFV] Selected TOPK=24 POWW=2.0 (OOF MAE=0.010194)
[Chem-kNN/Tc] TOPK=2